Config

In [1]:
# Pandas
import pandas as pd
pd.set_option("display.max_columns", None)

# Numpy
import numpy as np

In [2]:
DATA_PATH = "UCI_Credit_Card.csv"

In [3]:
SEED = 42
FOLD = 3

# EDA

In [4]:
df_raw = pd.read_csv("UCI_Credit_Card.csv")
df_raw

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,1,3,1,39,0,0,0,0,0,0,188948.0,192815.0,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,0,0,1683.0,1828.0,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,1,2,2,37,4,3,2,-1,0,0,3565.0,3356.0,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,1,3,1,41,1,-1,0,0,0,-1,-1645.0,78379.0,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [5]:
df_raw[df_raw.duplicated()]

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month


In [6]:
df_raw.isnull().sum()

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64

In [7]:
PAY = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
BILL = [f'BILL_AMT{i}' for i in range(1, 7)]
PAID = [f'PAY_AMT{i}' for i in range(1, 7)]
PROFILE = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE']
RAW = PROFILE + PAY + BILL + PAID
TARGET = 'default.payment.next.month'

In [8]:
df_raw

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,1,3,1,39,0,0,0,0,0,0,188948.0,192815.0,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,0,0,1683.0,1828.0,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,1,2,2,37,4,3,2,-1,0,0,3565.0,3356.0,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,1,3,1,41,1,-1,0,0,0,-1,-1645.0,78379.0,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


# FEATURE ENGINEERING

Dựa trên khung phân tích khách hàng RFM (Recency–Frequency–Monetary), được điều chỉnh cho dữ liệu tín dụng, ta xem mỗi dòng là một khách hàng với lịch sử thanh toán và dư nợ trong sáu tháng. Từ lịch sử này, các đặc trưng về thời điểm gần nhất, tần suất và giá trị tiền tệ được xây dựng để dự đoán khả năng khách hàng có hoặc không vỡ nợ thanh toán trong tháng tiếp theo.

In [9]:
df_copy = df_raw.copy()

**Recency:** Mức độ gần đây của hành vi thanh toán/trễ thanh toán
- `r_months_since_delay`: Lần trễ thanh toán gần nhất xảy ra cách đây bao nhiêu tháng.

In [10]:
delay = df_copy[PAY].to_numpy() > 0

df_copy["r_months_since_delay"] = np.where(delay.any(axis=1), delay.argmax(axis=1), 6)

**Frequency**: Tần suất xuất hiện tình trạng trễ
+ `f_total_delay_months`: Mức độ thường xuyên trễ thanh toán.

In [11]:
df_copy["f_total_delay_months"] = delay.sum(axis=1)

**Monetary:** Quy mô thanh toán so với số dư sao kê.
- `m_amt_bill`: Tổng PAY_AMT1..5 / tổng phần dương BILL_AMT2..6, ghép thanh toán với sao kê tháng trước.
- Đây là tỷ lệ tổng hợp qua các kỳ, không phải tỷ lệ tổng nợ đã trả hết. Mẫu số 0 trả NaN; giữ tỷ lệ vượt 1.

In [12]:
sum_bill = df_copy[BILL[1:]].clip(lower=0).sum(axis=1)
sum_pay = df_copy[PAID[:5]].sum(axis=1)

df_copy["m_amt_bill"] = sum_pay / sum_bill.where(sum_bill > 0)

In [13]:
df_copy

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month,r_months_since_delay,f_total_delay_months,m_amt_bill
0,1,20000.0,2,2,1,24,2,2,-1,-1,-2,-2,3913.0,3102.0,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1,0,2,0.181746
1,2,120000.0,2,2,2,26,-1,2,0,0,0,2,2682.0,1725.0,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1,1,2,0.208406
2,3,90000.0,2,2,2,34,0,0,0,0,0,0,29239.0,14027.0,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0,6,0,0.083105
3,4,50000.0,2,2,1,37,0,0,0,0,0,0,46990.0,48233.0,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0,6,0,0.040077
4,5,50000.0,1,2,1,57,-1,0,-1,0,0,0,8617.0,5670.0,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0,6,0,0.579516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,1,3,1,39,0,0,0,0,0,0,188948.0,192815.0,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0,6,0,0.077461
29996,29997,150000.0,1,3,2,43,-1,-1,-1,-1,0,0,1683.0,1828.0,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0,6,0,0.743115
29997,29998,30000.0,1,2,2,37,4,3,2,-1,0,0,3565.0,3356.0,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1,0,3,0.421329
29998,29999,80000.0,1,3,1,41,1,-1,0,0,0,-1,-1645.0,78379.0,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1,0,1,0.541934


# MODEL

**Cách chia:**
Chia dataset ra thành 3 phần 60/20/20.
+ 60% train: dùng cross-validation 3 fold để chọn mô hình và siêu tham số. Sau khi chọn xong, huấn luyện lại mô hình trên toàn bộ 60% này.
+ 20% validation: dùng để chọn ngưỡng phân loại phù hợp, chẳng hạn ưu tiên phát hiện khách hàng default nhiều hơn.
+ 20% test: dùng để đánh giá cuối cùng sau khi đã chốt mô hình, siêu tham số và ngưỡng. Không dùng kết quả test để điều chỉnh tiếp.

In [14]:
from sklearn.model_selection import train_test_split, StratifiedKFold

RFM = ["r_months_since_delay", "f_total_delay_months", "m_amt_bill"]
X = df_copy[RAW + RFM]
y = df_copy[TARGET]

In [15]:
# Chia train , val, test
X_model, X_test, y_model, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, stratify=y, random_state=SEED, shuffle=True)

X_train, X_val, y_train, y_val = train_test_split(X_model, y_model, train_size=0.75, test_size=0.25, stratify=y_model, random_state=SEED, shuffle=True)

# K FOLD
cv = StratifiedKFold(n_splits=FOLD, random_state=SEED, shuffle=True)

In [23]:
split_summary = pd.DataFrame([
    {"split": name, "rows": len(labels), "share": len(labels) / len(y),
     "default_rate": labels.mean()}
    for name, labels in [("train", y_train), ("validation", y_val), ("test", y_test)]
])

split_summary

,split,rows,share,default_rate
0,train,18000,0.6,0.221222
1,validation,6000,0.2,0.221167
2,test,6000,0.2,0.221167


## 1. Kiểm tra các tập dữ liệu
Giữ cách chia ngẫu nhiên có phân tầng 60/20/20 hiện tại. Các tập không trùng index; cách chia này chưa ràng buộc các bản ghi tương tự/trùng predictor phải cùng nhóm.
Các đặc trưng RFM chỉ dùng lịch sử trong từng dòng nên có thể tính trước khi chia. Imputer, encoder và scaler được học trong pipeline của mỗi fold.

## 2. Pipeline tiền xử lý
- Biến số: điền thiếu bằng median; thêm RobustScaler cho Logistic Regression.
- Biến phân loại: điền bằng mode và one-hot; các mã PAY được xem là phân loại.
- Model cây không cần scaling. Dùng one-hot dạng dense để tương thích HistGradientBoosting.

Mỗi pipeline được fit riêng trong từng fold, tránh học tiền xử lý từ phần đánh giá.

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"] + PAY
num_cols = [column for column in X_train.columns if column not in cat_cols]


def make_pipeline(model, scale=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scaler", RobustScaler()))

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", Pipeline(numeric_steps), num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ])
    return Pipeline([("preprocess", preprocessor), ("model", model)])

## 3. Chọn model và siêu tham số bằng StratifiedKFold
Mỗi model thử 4 cấu hình trên cùng 3 fold của **60% train**. Dùng AP làm tiêu chí chọn, ROC-AUC để tham khảo. GridSearchCV tự huấn luyện và đánh giá từng fold; `refit="AP"` fit lại cấu hình tốt nhất trên toàn bộ train.

Chọn model có AP CV trung bình cao nhất. Không dùng validation hoặc test để chọn model. Giới hạn chạy song song ở 2 tác vụ để tránh chiếm quá nhiều tài nguyên.

In [18]:
model_specs = {
    "Logistic Regression": (
        make_pipeline(LogisticRegression(max_iter=3000, random_state=SEED), scale=True),
        {"model__C": [0.1, 1.0], "model__class_weight": [None, "balanced"]},
    ),
    "Random Forest": (
        make_pipeline(RandomForestClassifier(n_estimators=120, random_state=SEED, n_jobs=1)),
        {"model__min_samples_leaf": [1, 5], "model__max_depth": [None, 12]},
    ),
    "HistGradientBoosting": (
        make_pipeline(HistGradientBoostingClassifier(
            max_iter=100, early_stopping=False, random_state=SEED,
        )),
        {"model__max_leaf_nodes": [15, 31], "model__l2_regularization": [0.0, 10.0]},
    ),
}

searches = {}
cv_rows = []
for name, (pipeline, param_grid) in model_specs.items():
    print(f"Đang chạy 3-fold CV: {name}", flush=True)
    search = GridSearchCV(
        pipeline, param_grid=param_grid,
        scoring={"AP": "average_precision", "ROC_AUC": "roc_auc"},
        refit="AP", cv=cv, n_jobs=2, error_score="raise",
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best_idx = search.best_index_
    cv_rows.append({
        "model": name,
        "AP_mean": search.cv_results_["mean_test_AP"][best_idx],
        "AP_std": search.cv_results_["std_test_AP"][best_idx],
        "ROC_AUC_mean": search.cv_results_["mean_test_ROC_AUC"][best_idx],
        "best_params": search.best_params_,
    })

cv_summary = pd.DataFrame(cv_rows).sort_values("AP_mean", ascending=False).reset_index(drop=True)
best_name = cv_summary.loc[0, "model"]
final_model = searches[best_name].best_estimator_
display(cv_summary)
print("Model được chọn bằng CV:", best_name)

Đang chạy 3-fold CV: Logistic Regression


Đang chạy 3-fold CV: Random Forest


Đang chạy 3-fold CV: HistGradientBoosting


,model,AP_mean,AP_std,ROC_AUC_mean,best_params
0,HistGradientBoosting,0.559923,0.007367,0.783072,"{'model__l2_regularization': 10.0, 'model__max..."
1,Random Forest,0.555486,0.009944,0.780532,"{'model__max_depth': 12, 'model__min_samples_l..."
2,Logistic Regression,0.548389,0.004773,0.775488,"{'model__C': 0.1, 'model__class_weight': None}"


Model được chọn bằng CV: HistGradientBoosting


## 4. Chọn ngưỡng trên 20% validation
Chưa có yêu cầu recall hoặc chi phí cảnh báo sai cụ thể, nên dùng **F1 cao nhất trên validation** làm quy tắc mặc định. Đây là lựa chọn thực nghiệm; không phải ngưỡng tối ưu cho mọi mục tiêu tín dụng.

Chỉ chọn ngưỡng cho model đã được chọn bằng CV. Nếu nhiều ngưỡng cùng đạt F1 cao nhất, chọn ngưỡng cao nhất. Giữ nguyên model đã fit trên 60% train; không fit thêm trên validation sau khi chọn ngưỡng.

In [19]:
val_proba = final_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_proba)

# Phần tử cuối của precision/recall không có ngưỡng tương ứng.
val_f1 = np.divide(
    2 * precision[:-1] * recall[:-1],
    precision[:-1] + recall[:-1],
    out=np.zeros_like(thresholds),
    where=(precision[:-1] + recall[:-1]) > 0,
)
threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1],
    "F1": val_f1,
})
threshold_results = threshold_results.sort_values(
    ["F1", "threshold"], ascending=[False, False],
).reset_index(drop=True)
best_threshold = float(threshold_results.loc[0, "threshold"])

# Kiểm tra F1 tính từ đường precision-recall khớp với dự đoán tại ngưỡng chọn.
assert np.isclose(
    f1_score(y_val, val_proba >= best_threshold), threshold_results.loc[0, "F1"],
)
display(threshold_results.head(10))
print(f"Ngưỡng chọn trên validation: {best_threshold:.4f}")

,threshold,precision,recall,F1
0,0.291174,0.542299,0.565185,0.553506
1,0.290007,0.541456,0.565938,0.553427
2,0.291612,0.542754,0.564431,0.553380
3,0.290736,0.541908,0.565185,0.553301
4,0.289982,0.541066,0.565938,0.553223
5,0.291489,0.542361,0.564431,0.553176
6,0.290446,0.541516,0.565185,0.553097
7,0.291637,0.542816,0.563677,0.553050
8,0.289893,0.540677,0.565938,0.553019
9,0.291274,0.541968,0.564431,0.552972


Ngưỡng chọn trên validation: 0.2912


## 5. Đánh giá cuối trên 20% test
Chỉ đánh giá model đã chọn và ngưỡng đã chốt. Báo cáo thêm ngưỡng 0.5 làm mốc tham khảo; không chọn lại ngưỡng hoặc model bằng kết quả test.

AP và ROC-AUC đo chất lượng xếp hạng xác suất nên không đổi theo ngưỡng. Precision, recall và F1 phụ thuộc ngưỡng.

In [20]:
test_proba = final_model.predict_proba(X_test)[:, 1]
test_rows = []
for label, threshold in [("Mốc 0.5", 0.5), ("Ngưỡng chọn từ validation", best_threshold)]:
    prediction = (test_proba >= threshold).astype(int)
    test_rows.append({
        "setting": label, "threshold": threshold,
        "AP": average_precision_score(y_test, test_proba),
        "ROC_AUC": roc_auc_score(y_test, test_proba),
        "accuracy": accuracy_score(y_test, prediction),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
    })
test_results = pd.DataFrame(test_rows)
display(test_results)

test_pred = (test_proba >= best_threshold).astype(int)
print(f"{best_name} — ngưỡng {best_threshold:.4f}")
print(classification_report(
    y_test, test_pred, labels=[0, 1],
    target_names=["Không default", "Default"], zero_division=0,
))
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
assert cm.sum() == len(y_test)
ConfusionMatrixDisplay(cm, display_labels=["Không default", "Default"]).plot(cmap="Blues")
plt.title("Test — model và ngưỡng đã chốt")
plt.tight_layout()
plt.show()

,setting,threshold,AP,ROC_AUC,accuracy,precision,recall,F1
0,Mốc 0.5,0.500000,0.553486,0.776595,0.817500,0.662921,0.355690,0.462972
1,Ngưỡng chọn từ validation,0.291174,0.553486,0.776595,0.789667,0.522807,0.561417,0.541424


HistGradientBoosting — ngưỡng 0.2912
               precision    recall  f1-score   support

Không default       0.87      0.85      0.86      4673
      Default       0.52      0.56      0.54      1327

     accuracy                           0.79      6000
    macro avg       0.70      0.71      0.70      6000
 weighted avg       0.80      0.79      0.79      6000

